In [1]:
# Brian Loch (4/26/2026)
# Packages
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

In [2]:
def predict_type(model, X, threshold=0.28):
    '''wrapper for entire prediction method that includes seperate threshold for ensuring that num types is either 1 or 2'''
    # later add extra func to handle list inputs (for user compatability)

    # Get raw probabilities (shape: n_samples, 18_types)
    probs = np.array([p[:, 1] for p in model.predict_proba(X)]).T

    # Find the index of the highest and second-highest prob for every row
    top_2_indices = np.argsort(probs, axis=1)[:, -2:]

    # Force the #1 choice to 1
    y_pred_constrained = np.zeros_like(probs)
    for i in range(len(y_pred_constrained)):
        # Top choice
        y_pred_constrained[i, top_2_indices[i, -1]] = 1 
        
        # Force the #2 choice ONLY if it's strong
        if probs[i, top_2_indices[i, -2]] > threshold:
            y_pred_constrained[i, top_2_indices[i, -2]] = 1 
    
    return y_pred_constrained

def calc_acc(Y_pred, Y_test):
    accuracy = accuracy_score(Y_test, Y_pred)
    pred_counts = y_pred_constrained.sum(axis=1)
    return accuracy

In [ ]:
# --- load prepared data ---
pkmn_df = pd.read_csv('../data/preprocessed_pokemon_data.csv')

Seed = 42

# Set indep vars to be everything that doesn't contain 'type'
indep_vars = [col for col in pkmn_df.columns if 'type' not in col.lower()]

# Target vars are the individual one-hot columns (e.g., 'type_Fire', 'type_Water')
dep_vars = [col for col in pkmn_df.columns if 'type' in col.lower()]

# threshold, chosen based on max acc across multiple trials
threshold = 0.28

X = pkmn_df[indep_vars]
Y = pkmn_df[dep_vars]

# --- Split and Train ---
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=Seed, shuffle=True
)

# Random Forest handles multi-output (one-hot Y) automatically
model_multi = RFC(n_estimators=100, random_state=Seed)
model_multi.fit(X_train, Y_train)

Y_pred = predict_type(model_multi, X_test)
acc = calc_acc(Y_pred, Y_test)

print(f'accuracy: {acc}')